# 📦 Amazon Product Reviews - Sentiment Analysis & Business Intelligence
### Course: AI Data Analysis (Horus Program - Growth Level Academy)
### Instructor: Eng. Aya Badwy
---
This notebook demonstrates the complete **End-to-End Machine Learning Pipeline**:
1. **Problem Definition & Data Ingestion**
2. **Exploratory Data Analysis (EDA) & Visualizations**
3. **Text Preprocessing & Feature Extraction (TF-IDF)**
4. **Model Comparison & 5-Fold Cross Validation**
5. **Hyperparameter Tuning & Evaluation Metrics**
6. **Model Persistence via Joblib**

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Data Ingestion & Inspection

In [ ]:
df = pd.read_csv('../data/raw/amazon_reviews_raw.csv')
print('Raw Data Shape:', df.shape)
df.head(3)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Star Rating Distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='Score', palette='viridis')
plt.title('Distribution of Star Ratings (1-5 Stars)')
plt.xlabel('Star Rating')
plt.ylabel('Review Count')
plt.show()

## 3. Data Cleaning & Feature Preparation

In [ ]:
# Filter out neutral (Score == 3) for clean binary sentiment
df_clean = df[df['Score'] != 3].copy()
df_clean['Sentiment'] = (df_clean['Score'] > 3).astype(int)

print('Positive reviews:', (df_clean['Sentiment'] == 1).sum())
print('Negative reviews:', (df_clean['Sentiment'] == 0).sum())

## 4. Model Training & Comparison

In [ ]:
# Load our processed balanced dataset
processed = pd.read_csv('../data/processed/cleaned_reviews.csv')
X = processed['Cleaned_Text']
y = processed['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=8000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42))
model.fit(X_train_vec, y_train)

preds = model.predict(X_test_vec)
print(classification_report(y_test, preds))

## 5. Model Saving via Joblib

In [ ]:
joblib.dump(model, '../models/sentiment_model.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print('Model and Vectorizer saved successfully!')